# Project 2 — Notebook 03: Cohort Retention Matrix

**Week 2 goal:** build the retention count matrix and retention percentage matrix.

The rows are customer start months (`cohort_month`) and the columns are how many months have passed since the first purchase (`cohort_index`).

## Step 1 — Import and load cohort data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)

import pandas as pd
import numpy as np

from src.data_prep import load_cohort_dataset, build_cohort_count_matrix, build_retention_matrix

pd.set_option('display.max_columns', 20)

In [ ]:
df = load_cohort_dataset(DATA_DIR)
print('rows:', f'{len(df):,}')
print('customers:', f'{df["customer_id"].nunique():,}')
df[['customer_id', 'transaction_month', 'cohort_month', 'cohort_index']].head()

## Step 2 — Build the cohort count matrix

Each value means: “How many unique customers from this cohort bought something in this later month?”

In [ ]:
cohort_matrix = build_cohort_count_matrix(df)
cohort_matrix

## Step 3 — Build the retention percentage matrix

Month 0 is always 100% because every customer is active in their first purchase month. For later months, I divide by the Month 0 cohort size.

In [ ]:
retention_matrix = build_retention_matrix(cohort_matrix)
retention_percent = (retention_matrix * 100).round(1)
retention_percent

## Step 4 — Validation checks

In [ ]:
month_zero_all_100 = (retention_matrix[0].round(4) == 1).all()
cohort_size_match = cohort_matrix[0].sum() == df['customer_id'].nunique()
negative_values = (retention_matrix.fillna(0) < 0).sum().sum()
over_100_values = (retention_matrix.fillna(0) > 1).sum().sum()

print('Month 0 is 100% for every cohort:', month_zero_all_100)
print('Month 0 customers = total customers:', cohort_size_match)
print('Negative retention values:', negative_values)
print('Retention values over 100%:', over_100_values)

print('\nAverage Month 1 retention:', f'{retention_matrix[1].dropna().mean() * 100:.1f}%')
print('Average Month 2 retention:', f'{retention_matrix[2].dropna().mean() * 100:.1f}%')

## Step 5 — Save the matrices

In [ ]:
cohort_matrix.to_csv(OUTPUT_DIR / 'cohort_count_matrix.csv')
retention_percent.to_csv(OUTPUT_DIR / 'retention_percentage_matrix.csv')

print('Saved:', OUTPUT_DIR / 'cohort_count_matrix.csv')
print('Saved:', OUTPUT_DIR / 'retention_percentage_matrix.csv')

## Week 2 notes

Retention drops a lot after the first month. The average Month 1 retention is only about one-fifth of customers. The heatmap in Week 4 will make this pattern easier to see.